# Export Telegram Messages

This notebook logs in with your Telegram user account via Telethon and exports text-only messages to JSONL with UTC timestamps.

In [ ]:
import json
import os
import asyncio
from datetime import timezone
from pathlib import Path

from dotenv import load_dotenv
from telethon import TelegramClient, utils
from telethon.errors import FloodWaitError, SessionPasswordNeededError

# Defaults requested in plan
CHAT_TITLE = ""
CHAT_ID = None  # Set this to a numeric chat id to bypass title lookup if duplicates exist
OUTPUT_PATH = "exports/messages.jsonl"

# Manual fallback if interactive prompt is not available in your VS Code notebook UI.

LOGIN_CODE_OVERRIDE = None # Replace with the code you receive from Telegram when logging in (available when running the last cell.)
# To request a new code: run on the terminal `rm -rf .telegram_sessions/` and then run the last cell again.
TWO_FA_PASSWORD_OVERRIDE = None


In [ ]:
# Resolve project root so notebook can run either from repo root or notebooks/
cwd = Path.cwd().resolve()
if (cwd / ".env").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / ".env").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd
ENV_PATH = PROJECT_ROOT / ".env"

if not ENV_PATH.exists():
    raise FileNotFoundError(
        "Missing .env file in project root. Copy .env.example to .env and fill TG_API_ID/TG_API_HASH/TG_PHONE/TG_CHAT_TITLE."
    )

load_dotenv(ENV_PATH)

required = ["TG_API_ID", "TG_API_HASH", "TG_PHONE", "TG_CHAT_TITLE"]
missing = [key for key in required if not os.getenv(key)]
if missing:
    raise ValueError(f"Missing required environment variables in .env: {', '.join(missing)}")

API_ID = int(os.getenv("TG_API_ID"))
API_HASH = os.getenv("TG_API_HASH")
PHONE = os.getenv("TG_PHONE")
CHAT_TITLE = os.getenv("TG_CHAT_TITLE", "").strip()
SESSION_NAME = os.getenv("TG_SESSION_NAME", "telegram_session")
SESSION_DIR = PROJECT_ROOT / ".telegram_sessions"
SESSION_DIR.mkdir(parents=True, exist_ok=True)
SESSION_PATH = SESSION_DIR / SESSION_NAME

OUTPUT_FILE = Path(OUTPUT_PATH)
if not OUTPUT_FILE.is_absolute():
    OUTPUT_FILE = (PROJECT_ROOT / OUTPUT_FILE).resolve()
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

def safe_path_display(path: Path) -> str:
    try:
        return str(path.resolve().relative_to(PROJECT_ROOT))
    except ValueError:
        return path.name

print("Using env: .env")
print(f"Session file: .telegram_sessions/{SESSION_NAME}.session")
print(f"Output file: {safe_path_display(OUTPUT_FILE)}")


In [ ]:
async def prompt_notebook_value(
    prompt: str,
    secret: bool = False,
    timeout_seconds: int = 25,
    override_value: str | None = None,
) -> str:
    """Prompt inline in Jupyter via ipywidgets. If widgets fail, require manual override."""
    if override_value:
        return str(override_value).strip()

    try:
        import ipywidgets as widgets
        from IPython.display import display

        event = asyncio.Event()
        holder = {"value": None}

        field_cls = widgets.Password if secret else widgets.Text
        field = field_cls(
            description=prompt,
            placeholder="Paste value, then press Enter or click Submit",
            layout=widgets.Layout(width="700px"),
        )
        button = widgets.Button(description="Submit", button_style="primary")
        feedback = widgets.HTML("")
        box = widgets.VBox([field, button, feedback])
        display(box)

        def _submit(_=None):
            value = (field.value or "").strip()
            if not value:
                feedback.value = "<span style='color:#b00;'>Value is required.</span>"
                return
            holder["value"] = value
            if not event.is_set():
                event.set()

        button.on_click(_submit)
        if hasattr(field, "on_submit"):
            try:
                field.on_submit(_submit)
            except Exception:
                pass

        print(f"{prompt}: waiting for inline input...")
        loop = asyncio.get_running_loop()
        start = loop.time()
        last_value = ""
        stable_since = None
        while not event.is_set():
            current = (field.value or "").strip()
            now = loop.time()

            if current:
                if current != last_value:
                    last_value = current
                    stable_since = now
                elif stable_since is not None and (now - stable_since) >= 0.8:
                    holder["value"] = current
                    event.set()
                    break

            if (now - start) > timeout_seconds:
                box.close()
                raise TimeoutError("Inline widget did not submit in time.")

            await asyncio.sleep(0.15)

        box.close()
        return holder["value"]
    except Exception as exc:
        raise RuntimeError(
            f"Could not collect '{prompt}' interactively. "
            "Set LOGIN_CODE_OVERRIDE (and TWO_FA_PASSWORD_OVERRIDE if needed) in the first cell, then rerun."
        ) from exc


async def ensure_login(client: TelegramClient, phone: str) -> None:
    await client.connect()
    if await client.is_user_authorized():
        print("Already authorized with existing session.")
        return

    print("Telegram login required. Sending code...")
    await client.send_code_request(phone)
    code = await prompt_notebook_value("Telegram login code", override_value=LOGIN_CODE_OVERRIDE)
    try:
        await client.sign_in(phone=phone, code=code)
    except SessionPasswordNeededError:
        password = await prompt_notebook_value(
            "Telegram 2FA password",
            secret=True,
            override_value=TWO_FA_PASSWORD_OVERRIDE,
        )
        await client.sign_in(password=password)

    print("Login complete; session persisted locally.")


async def resolve_chat_entity(client: TelegramClient, chat_title: str, chat_id=None):
    if chat_id is not None:
        return await client.get_entity(chat_id)

    matches = []
    async for dialog in client.iter_dialogs():
        if dialog.name == chat_title:
            peer_id = utils.get_peer_id(dialog.entity)
            username = getattr(dialog.entity, "username", None)
            matches.append((dialog.entity, dialog.name, peer_id, username))

    if not matches:
        raise RuntimeError(f"No chat found with exact title: {chat_title!r}")

    if len(matches) > 1:
        print(f"Multiple chats found with title {chat_title!r}. Use CHAT_ID to disambiguate:")
        for idx, (_, title, peer_id, username) in enumerate(matches, start=1):
            print(f"  {idx}. title={title!r}, peer_id={peer_id}, username={username}")
        raise RuntimeError("Ambiguous chat title. Set CHAT_ID and rerun.")

    entity, _, peer_id, username = matches[0]
    print(f"Selected chat peer_id={peer_id}, username={username}")
    return entity


def to_utc_iso(dt):
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")


async def export_messages_text_only(client: TelegramClient, entity, output_file: Path):
    scanned = 0
    written = 0
    offset_id = 0

    with output_file.open("w", encoding="utf-8") as fh:
        while True:
            batch_seen = 0
            try:
                async for msg in client.iter_messages(entity, limit=1000, offset_id=offset_id, reverse=False):
                    scanned += 1
                    batch_seen += 1
                    offset_id = msg.id

                    text = (msg.message or "").strip()
                    if not text:
                        continue
                    if msg.media is not None:
                        # Skip captions/media to keep export strictly text-only messages
                        continue

                    record = {
                        "sent_at": to_utc_iso(msg.date),
                        "text": text,
                    }
                    fh.write(json.dumps(record, ensure_ascii=False) + "\n")
                    written += 1

                if batch_seen == 0:
                    break

            except FloodWaitError as err:
                wait_seconds = int(getattr(err, "seconds", 0)) + 1
                print(f"FloodWait triggered. Sleeping {wait_seconds} seconds...")
                await asyncio.sleep(wait_seconds)

    return scanned, written


In [ ]:
async def run_export():
    client = TelegramClient(str(SESSION_PATH), API_ID, API_HASH)

    try:
        await ensure_login(client, PHONE)
        entity = await resolve_chat_entity(client, CHAT_TITLE, CHAT_ID)

        scanned, written = await export_messages_text_only(client, entity, OUTPUT_FILE)
        print(f"Done. Scanned {scanned} messages, exported {written} text messages.")

        first_line = None
        last_line = None
        with OUTPUT_FILE.open("r", encoding="utf-8") as fh:
            for line in fh:
                line = line.rstrip("\n")
                if first_line is None:
                    first_line = line
                last_line = line

        if first_line is not None:
            print("Newest exported line (first line):")
            print(first_line)
            print("Oldest exported line (last line):")
            print(last_line)
        else:
            print("Output file is empty (no matching text-only messages).")
    finally:
        await client.disconnect()

await run_export()
